# KDEWatershedClusterer

Lab notebook ถอด `KDEWatershedClusterer` (อ้างอิงจาก `old-concept-code/clustering.py`) ออกมาดูเดี่ยว ๆ แยกจาก pipeline หลัก (ไม่มี embedding/UMAP) — ใช้ synthetic 2D points เพื่อดู logic ภายในทีละขั้น

## Logic โดยรวม

`KDEWatershedClusterer` จับกลุ่มจุด 2 มิติโดยมองความหนาแน่นของจุดเป็น "ภูมิประเทศ" (terrain) แล้วแบ่งกลุ่มตามลักษณะภูมิประเทศนั้น มี 4 ขั้นตอนหลัก:

1. **KDE (Kernel Density Estimation)** — ประมาณค่าความหนาแน่นของจุดทั่วพื้นที่ 2 มิติ ด้วย `scipy.stats.gaussian_kde` แล้ว sample ค่าความหนาแน่นลงบน grid (`grid_size` × `grid_size`) ผลลัพธ์คือ "เนินเขา" ตรงที่จุดหนาแน่น และ "หุบเขา" ตรงที่จุดเบาบาง
2. **Peak finding** — หาตำแหน่งยอดเขา (local maxima) บน grid ความหนาแน่นด้วย `scipy.ndimage.maximum_filter` (เทียบค่าตัวเองกับเพื่อนบ้านในหน้าต่างขนาด `neighborhood_size`) แล้วกรองยอดที่เตี้ยเกินไปทิ้งด้วย `rel_threshold` (สัดส่วนเทียบกับยอดสูงสุด) — ยอดที่เหลือคือ **cluster seed** แต่ละยอด = 1 cluster ที่เป็นไปได้
3. **Watershed segmentation (พร้อม ridge line)** — เปรียบเหมือนปล่อยน้ำท่วมจาก "ยอดเขากลับหัว" (ใช้ `-kde_values` เพื่อให้ยอดกลายเป็นแอ่ง) แต่ละยอดที่เจอใน step 2 เป็นจุดเริ่ม (`markers_grid`) น้ำจากแต่ละแอ่งจะขยายจนชนสันเขา (ridge) กับแอ่งข้างเคียง — ใช้ `watershed_line=True` ทำให้สันเขาได้ label `0` แยกออกมาจาก basin จริง (`labels_grid`) แทนที่จะยัดทุก grid cell เข้า basin ใดบาสินหนึ่งเสมอ
4. **Map จุดจริงกลับ + ตรวจ boundary/bridge** — เอาพิกัดจริงของแต่ละจุด (`xy`) map กลับไปดูว่าตกอยู่ grid cell ไหน แล้วเปิดหน้าต่างเล็ก ๆ (`bridge_window` × `bridge_window`) รอบ cell นั้นดูว่าเจอ basin กี่แบบ — เจอ basin เดียว = จุดนั้นอยู่ในกลุ่มนั้นชัดเจน, เจอมากกว่า 1 basin = จุดนั้นเป็น **bridge point** (อยู่ระหว่างยอดเขาตั้งแต่ 2 ลูกขึ้นไป เช่น จุดในหุบเขา/แอ่งน้ำที่ซอกแซกอยู่ตีนเขาหลายลูก) ไม่ถูกยัดเข้ากลุ่มไหนอีกต่อไป

จุดที่ไม่มี basin ให้ยึดเลย (ทั้ง dataset ไม่มี peak) จะได้ label `-1` (noise) ส่วน **bridge point** จะได้ label `-2` พร้อม `bridge_between` บอกว่าอยู่ระหว่างกลุ่มไหนบ้าง (ดูรายละเอียดที่มาของแนวคิดนี้ที่ `docs/hierarchical-taxonomy-concept.md` section 11)

### Parameter สำคัญ

- **`bw_method`** — bandwidth ของ KDE คุมความ "เรียบ" ของภูมิประเทศความหนาแน่น: เล็ก → เนินแคบ ยอดเยอะ (cluster ย่อยเยอะ), ใหญ่ → เนินกว้าง ยอดถูกรวมกัน (cluster ใหญ่น้อยกลุ่ม)
- **`rel_threshold`** — ยอดที่ความสูง < `rel_threshold × max_density` จะถูกตัดทิ้ง (default = `bw_method`) กันไม่ให้ noise เล็ก ๆ กลายเป็น cluster ปลอม
- **`grid_size`** — ความละเอียดของ grid ที่ใช้คำนวณ KDE/watershed ยิ่งสูงยิ่งละเอียดแต่ช้าลง
- **`neighborhood_size`** — ขนาดหน้าต่างตอนหา local maxima ยิ่งใหญ่ยิ่งหายอดได้ยากขึ้น (กันยอดที่อยู่ใกล้กันเกินไปไม่ให้แยกเป็นคนละ cluster)
- **`bridge_window`** (ใหม่) — ขนาดหน้าต่าง (NxN) เช็ค label รอบจุด หา basin ข้างเคียง ยิ่งใหญ่ยิ่งจับ bridge point ไวขึ้น (แต่เสี่ยง flag เกินจำเป็นถ้าใหญ่ไป)
- **`bridge_rel_threshold`** (ใหม่, optional) — ถ้าตั้งค่า จะเสริม density-ratio check: จุดที่ยังหนาแน่นพอเทียบกับยอดที่เด่นสุดในหน้าต่าง (แค่ติดขอบ grid ไม่ได้อยู่ก้นแอ่งจริง) จะไม่ถูกนับเป็น bridge ปล่อยเป็น `None` (default) = ปิดไว้ ใช้แค่ label diversity ล้วน ๆ

In [ ]:
%pip install -q numpy scipy scikit-image scikit-learn matplotlib pandas

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
from scipy.ndimage import maximum_filter
from skimage.segmentation import watershed

np.random.seed(42)
import matplotlib.font_manager as fm

_THAI_FONT_CANDIDATES = [
    "Noto Sans Thai", "TH Sarabun New", "Sarabun", "Waree", "Loma", "Garuda",
    "Kinnari", "Norasi", "Sawasdee", "Tlwg Typist", "Tlwg Typo", "Tahoma", "Arial Unicode MS",
]


def _set_thai_font():
    """กัน UserWarning: Glyph ... missing from font(s) DejaVu Sans ตอน plot ข้อความไทย"""
    available = {f.name for f in fm.fontManager.ttflist}
    for name in _THAI_FONT_CANDIDATES:
        if name in available:
            plt.rcParams["font.family"] = name
            print(f"ใช้ฟอนต์ไทย: {name}")
            return name
    print(
        "\u26a0\ufe0f ไม่พบฟอนต์ไทยในระบบ ตัวอักษรไทยในกราฟจะแสดงเป็นกล่องว่าง (missing glyph)\n"
        "ติดตั้งฟอนต์ก่อน เช่น: sudo apt install fonts-thai-tlwg  (หรือ fonts-noto-cjk / Noto Sans Thai)\n"
        "แล้วล้าง matplotlib font cache: rm -rf ~/.cache/matplotlib  ก่อนรัน notebook ใหม่"
    )
    return None


_set_thai_font()

## 0. Synthetic 2D data

แทนที่จะรัน UMAP จริง ใช้จุด 2 มิติสังเคราะห์ (4 กลุ่มความหนาแน่นต่างกัน + noise เบาบาง) เพื่อดู behaviour ของ `KDEWatershedClusterer` ล้วน ๆ — ในงานจริง `xy` ตัวนี้คือ output ของ UMAP (`coordinate_xy`)

In [ ]:
from sklearn.datasets import make_blobs

blob_xy, blob_truth = make_blobs(
    n_samples=[120, 90, 60, 40],
    centers=[(0, 0), (5, 0.5), (2.5, 4), (6, 4.5)],
    cluster_std=[0.6, 0.5, 0.4, 0.7],
    random_state=42,
)

# noise เบาบางกระจายทั่วพื้นที่
noise = np.random.uniform(low=blob_xy.min(axis=0) - 1, high=blob_xy.max(axis=0) + 1, size=(15, 2))
xy = np.vstack([blob_xy, noise])

print("xy shape:", xy.shape)
plt.figure(figsize=(6, 5))
plt.scatter(xy[:, 0], xy[:, 1], s=20, alpha=0.7)
plt.title("Synthetic 2D points (input ของ KDEWatershedClusterer)")
plt.grid(True, linestyle="--", linewidth=0.5)
plt.show()

## 1. `KDEWatershedClusterer` — โค้ดเต็ม

copy ตรงจาก `old-concept-code/clustering.py` (ไม่แก้ logic) — เก็บไว้ใช้ทั้งแบบเรียก `fit_predict()` รวดเดียว และแบบเรียก method ภายในทีละขั้นใน section ถัดไป

In [ ]:
class KDEWatershedClusterer:
    """KDE density + watershed segmentation clusterer, with boundary/bridge point detection.

    อ้างอิงจาก old-concept-code/clustering.py + docs/hierarchical-taxonomy-concept.md (section 11)
    """

    def __init__(
        self,
        bw_method=0.1,
        grid_size=100,
        neighborhood_size=5,
        rel_threshold=None,
        bridge_window=3,
        bridge_rel_threshold=None,
    ):
        self.bw_method = bw_method
        self.grid_size = grid_size
        self.neighborhood_size = neighborhood_size
        self.rel_threshold = rel_threshold if rel_threshold is not None else self.bw_method
        self.bridge_window = bridge_window                  # ขนาดหน้าต่าง (NxN) เช็ค label รอบจุด หา basin ข้างเคียง
        self.bridge_rel_threshold = bridge_rel_threshold    # None = ปิด density-ratio check (ใช้แค่ label diversity)
        self.embedding_, self.kde_dict_, self.peaks_, self.cluster_centers_, self.labels_ = [None] * 5
        self.bridge_between_ = {}  # point index -> sorted list ของ cluster id ที่จุดนั้นอยู่ระหว่างกลาง

    def fit_predict(self, X):
        self.embedding_ = X
        x, y = X[:, 0], X[:, 1]
        self.kde_dict_ = self._calculate_kde(x, y)
        self.peaks_ = self._find_peaks()
        if self.peaks_:
            self.cluster_centers_ = np.array([[p["x"], p["y"]] for p in self.peaks_])
        else:
            self.cluster_centers_ = np.empty((0, 2))
        self.labels_ = self._assign_labels_by_watershed(X)
        return self.labels_

    def get_results_dict(self):
        if self.labels_ is None:
            raise RuntimeError("ต้องรัน .fit_predict(X) ก่อน")
        centers_dict = {str(i): c.tolist() for i, c in enumerate(self.cluster_centers_)}
        serializable_kde = {k: v.tolist() if isinstance(v, np.ndarray) else v for k, v in self.kde_dict_.items()}
        return {
            "labels": self.labels_.tolist(),
            "centers": centers_dict,
            "kde_dict": serializable_kde,
            "xy": self.embedding_.tolist(),
            "bridge_between": {str(k): v for k, v in self.bridge_between_.items()},
        }

    def _calculate_kde(self, x, y):
        kde = gaussian_kde(np.vstack([x, y]), bw_method=self.bw_method)
        xmin, xmax, ymin, ymax = x.min() - 1, x.max() + 1, y.min() - 1, y.max() + 1
        x_grid, y_grid = np.meshgrid(np.linspace(xmin, xmax, self.grid_size), np.linspace(ymin, ymax, self.grid_size))
        kde_values = kde(np.vstack([x_grid.ravel(), y_grid.ravel()])).reshape(x_grid.shape)
        return {"x_grid": x_grid, "y_grid": y_grid, "kde_values": kde_values, "bw_method": self.bw_method}

    def _find_peaks(self):
        vals = self.kde_dict_["kde_values"]
        mask = (vals == maximum_filter(vals, size=self.neighborhood_size)) & (vals > vals.max() * self.rel_threshold)
        rows, cols = np.where(mask)
        if len(rows) == 0:
            return []
        return [{"x": self.kde_dict_["x_grid"][r, c], "y": self.kde_dict_["y_grid"][r, c]} for r, c in zip(rows, cols)]

    def _assign_labels_by_watershed(self, xy):
        self.bridge_between_ = {}
        if not self.peaks_:
            return np.full(xy.shape[0], -1)

        kde_values = self.kde_dict_["kde_values"]
        xmin, xmax = self.kde_dict_["x_grid"][0, 0], self.kde_dict_["x_grid"][0, -1]
        ymin, ymax = self.kde_dict_["y_grid"][0, 0], self.kde_dict_["y_grid"][-1, 0]

        markers_grid = np.zeros_like(kde_values, dtype=int)
        for i, peak in enumerate(self.peaks_):
            r = np.abs(self.kde_dict_["y_grid"][:, 0] - peak["y"]).argmin()
            c = np.abs(self.kde_dict_["x_grid"][0, :] - peak["x"]).argmin()
            markers_grid[r, c] = i + 1

        # watershed_line=True: สันเขา (ridge) ระหว่าง basin ได้ label 0 แยกออกมา
        # แทนที่จะยัดทุก grid cell เข้า basin ใดบาสินหนึ่งเสมอเหมือนเดิม (ดู docs section 11)
        labels_grid = watershed(-kde_values, markers_grid, mask=np.ones_like(kde_values, dtype=bool), watershed_line=True)

        cols = np.clip(((xy[:, 0] - xmin) / (xmax - xmin) * (self.grid_size - 1)).astype(int), 0, self.grid_size - 1)
        rows = np.clip(((xy[:, 1] - ymin) / (ymax - ymin) * (self.grid_size - 1)).astype(int), 0, self.grid_size - 1)

        half_w = self.bridge_window // 2
        labels = np.full(xy.shape[0], -1)

        for idx, (r, c) in enumerate(zip(rows, cols)):
            r0, r1 = max(0, r - half_w), min(self.grid_size, r + half_w + 1)
            c0, c1 = max(0, c - half_w), min(self.grid_size, c + half_w + 1)
            window = labels_grid[r0:r1, c0:c1]
            neighbor_ids = sorted(int(v) for v in np.unique(window) if v > 0)  # ตัด 0 (ridge) ทิ้ง เหลือแต่ basin จริง

            if len(neighbor_ids) == 0:
                labels[idx] = -1  # ไม่มี basin ให้ยึดเลยแม้แต่ในหน้าต่างรอบตัว (rare)
            elif len(neighbor_ids) == 1:
                labels[idx] = neighbor_ids[0] - 1  # อยู่แน่นอนใน basin เดียว -> cluster ปกติ
            else:
                # เจอมากกว่า 1 basin ในหน้าต่างรอบจุด -> ผู้สมัคร bridge point
                cluster_ids = [n - 1 for n in neighbor_ids]
                peak_densities = {}
                for cid in cluster_ids:
                    peak = self.peaks_[cid]
                    rr = np.abs(self.kde_dict_["y_grid"][:, 0] - peak["y"]).argmin()
                    cc = np.abs(self.kde_dict_["x_grid"][0, :] - peak["x"]).argmin()
                    peak_densities[cid] = kde_values[rr, cc]

                if self.bridge_rel_threshold is not None:
                    # density-ratio เสริม: ถ้าความหนาแน่น ณ จุดนี้ยังสูงพอเทียบกับยอดที่เด่นสุดในหน้าต่าง
                    # (แค่ติดขอบ grid ไม่ได้อยู่ก้นแอ่งจริง) ให้ fallback เข้ากลุ่มนั้นแทนการเป็น bridge
                    point_density = kde_values[r, c]
                    best_cid = max(peak_densities, key=peak_densities.get)
                    if point_density >= self.bridge_rel_threshold * peak_densities[best_cid]:
                        labels[idx] = best_cid
                        continue

                labels[idx] = -2
                self.bridge_between_[idx] = cluster_ids

        return labels

## 2. ไล่ทีละขั้น (internal step-by-step)

เรียก method ภายใน (`_calculate_kde`, `_find_peaks`, `_assign_labels_by_watershed`) ตรง ๆ เพื่อดูผลลัพธ์แต่ละขั้นก่อนถึง `fit_predict()`

In [ ]:
clusterer = KDEWatershedClusterer(bw_method=0.15, grid_size=120, neighborhood_size=7)

# --- Step 1: KDE ---
kde_dict = clusterer._calculate_kde(xy[:, 0], xy[:, 1])
clusterer.kde_dict_ = kde_dict  # เก็บไว้ให้ method ถัดไปใช้ต่อ เหมือนที่ fit_predict ทำ

fig, ax = plt.subplots(figsize=(6, 5))
cf = ax.contourf(kde_dict["x_grid"], kde_dict["y_grid"], kde_dict["kde_values"], levels=20, cmap="Blues")
ax.scatter(xy[:, 0], xy[:, 1], s=10, color="black", alpha=0.4)
fig.colorbar(cf, ax=ax, label="density")
ax.set_title(f"Step 1: KDE density surface (bw_method={clusterer.bw_method})")
plt.show()

In [ ]:
# --- Step 2: Peak finding ---
peaks = clusterer._find_peaks()
clusterer.peaks_ = peaks
print(f"peaks found: {len(peaks)}")

fig, ax = plt.subplots(figsize=(6, 5))
ax.contourf(kde_dict["x_grid"], kde_dict["y_grid"], kde_dict["kde_values"], levels=20, cmap="Blues", alpha=0.7)
ax.scatter(xy[:, 0], xy[:, 1], s=10, color="gray", alpha=0.4, label="points")
if peaks:
    px = [p["x"] for p in peaks]
    py = [p["y"] for p in peaks]
    ax.scatter(px, py, marker="x", s=100, c="red", linewidth=2, zorder=10, label="peak (cluster seed)")
ax.set_title("Step 2: Local maxima บน density surface = cluster seeds")
ax.legend()
plt.show()

In [ ]:
# --- Step 3: Watershed segmentation บน grid พร้อม ridge line (ก่อน map กลับไปที่จุดจริง) ---
kde_values = kde_dict["kde_values"]
markers_grid = np.zeros_like(kde_values, dtype=int)
for i, peak in enumerate(peaks):
    r = np.abs(kde_dict["y_grid"][:, 0] - peak["y"]).argmin()
    c = np.abs(kde_dict["x_grid"][0, :] - peak["x"]).argmin()
    markers_grid[r, c] = i + 1

# watershed_line=True -> สันเขา (ridge) ระหว่าง basin ได้ label 0 แยกออกมาชัดเจน
labels_grid = watershed(-kde_values, markers_grid, mask=np.ones_like(kde_values, dtype=bool), watershed_line=True)

fig, ax = plt.subplots(figsize=(6, 5))
ax.contourf(kde_dict["x_grid"], kde_dict["y_grid"], labels_grid, levels=len(peaks) + 1, cmap="tab10", alpha=0.6)
ax.contour(kde_dict["x_grid"], kde_dict["y_grid"], kde_values, levels=8, colors="black", linewidths=0.4, alpha=0.5)
if peaks:
    ax.scatter([p["x"] for p in peaks], [p["y"] for p in peaks], marker="x", s=100, c="red", zorder=10)
ax.set_title("Step 3: Watershed region ต่อ grid cell (สีที่ 0 = ridge/สันเขา) ก่อน map กลับไปยังจุดจริง")
plt.show()

In [ ]:
# --- Step 4: map พิกัดจุดจริง (xy) กลับไปหา region บน labels_grid + ตรวจ boundary/bridge -> label สุดท้ายต่อจุด ---
step_labels = clusterer._assign_labels_by_watershed(xy)

fig, ax = plt.subplots(figsize=(6, 5))
for cid in sorted(c for c in np.unique(step_labels) if c >= 0):
    pts = xy[step_labels == cid]
    ax.scatter(pts[:, 0], pts[:, 1], s=25, label=f"cluster {cid}", edgecolor="black", linewidth=0.3)
bridge_pts = xy[step_labels == -2]
if len(bridge_pts):
    ax.scatter(bridge_pts[:, 0], bridge_pts[:, 1], s=60, marker="D", facecolor="none", edgecolor="black", linewidth=1.5, label="bridge point (-2)")
noise_pts = xy[step_labels == -1]
if len(noise_pts):
    ax.scatter(noise_pts[:, 0], noise_pts[:, 1], s=25, c="lightgray", label="noise (-1)", edgecolor="black", linewidth=0.3)
ax.set_title("Step 4: label สุดท้ายต่อจุด (map xy -> watershed region + bridge check)")
ax.legend()
plt.show()

print(f"core: {int((step_labels >= 0).sum())} | bridge: {int((step_labels == -2).sum())} | noise: {int((step_labels == -1).sum())}")

## 3. `fit_predict()` แบบรวดเดียว

ผลควรตรงกับ step-by-step ด้านบน (`fit_predict` เรียก method ภายในลำดับเดียวกัน)

In [ ]:
clusterer2 = KDEWatershedClusterer(bw_method=0.15, grid_size=120, neighborhood_size=7)
labels = clusterer2.fit_predict(xy)

print("labels match step-by-step:", np.array_equal(labels, step_labels))

result_df = pd.DataFrame({"x": xy[:, 0], "y": xy[:, 1], "cluster_id": labels})
result_df["cluster_id"].value_counts().sort_index()

## 4. ผลของ `bw_method` (bandwidth)

เทียบ bandwidth เล็ก vs กลาง vs ใหญ่ บนข้อมูลชุดเดียวกัน — bandwidth เล็กทำให้ยอดเขาแคบและเยอะ (แบ่งย่อย รวม noise เป็น cluster ปลอมได้ง่าย), bandwidth ใหญ่ทำให้ยอดเขากว้างและถูกรวมกัน (cluster ใหญ่ น้อยกลุ่ม อาจรวมกลุ่มที่ควรแยกเข้าด้วยกัน)

In [ ]:
bw_values = [0.05, 0.15, 0.4]

fig, axes = plt.subplots(1, len(bw_values), figsize=(5 * len(bw_values), 5))
for ax, bw in zip(axes, bw_values):
    c = KDEWatershedClusterer(bw_method=bw, grid_size=120, neighborhood_size=7)
    lbl = c.fit_predict(xy)
    n_clusters = len([x for x in np.unique(lbl) if x >= 0])

    ax.contourf(c.kde_dict_["x_grid"], c.kde_dict_["y_grid"], c.kde_dict_["kde_values"], levels=20, cmap="Blues", alpha=0.6)
    for cid in sorted(x for x in np.unique(lbl) if x >= 0):
        pts = xy[lbl == cid]
        ax.scatter(pts[:, 0], pts[:, 1], s=20, edgecolor="black", linewidth=0.3)
    ax.set_title(f"bw_method={bw} -> {n_clusters} clusters")
plt.tight_layout()
plt.show()

## 5. ผลของ `rel_threshold`

`rel_threshold` กรองยอดที่เตี้ยกว่า `rel_threshold × max_density` ทิ้งก่อนเข้า watershed — ค่ายิ่งสูง ยิ่งเหลือแค่ยอดที่เด่นชัดจริง ๆ (คลัสเตอร์เล็ก/เบาบางถูกมองเป็น noise) ค่ายิ่งต่ำ ยิ่งรับยอดเล็ก ๆ เข้ามาเป็น cluster ง่ายขึ้น

In [ ]:
thresholds = [0.05, 0.3, 0.6]

fig, axes = plt.subplots(1, len(thresholds), figsize=(5 * len(thresholds), 5))
for ax, th in zip(axes, thresholds):
    c = KDEWatershedClusterer(bw_method=0.15, grid_size=120, neighborhood_size=7, rel_threshold=th)
    lbl = c.fit_predict(xy)
    n_clusters = len([x for x in np.unique(lbl) if x >= 0])
    n_noise = int((lbl == -1).sum())

    ax.contourf(c.kde_dict_["x_grid"], c.kde_dict_["y_grid"], c.kde_dict_["kde_values"], levels=20, cmap="Blues", alpha=0.6)
    for cid in sorted(x for x in np.unique(lbl) if x >= 0):
        pts = xy[lbl == cid]
        ax.scatter(pts[:, 0], pts[:, 1], s=20, edgecolor="black", linewidth=0.3)
    noise_pts = xy[lbl == -1]
    if len(noise_pts):
        ax.scatter(noise_pts[:, 0], noise_pts[:, 1], s=20, c="lightgray", edgecolor="black", linewidth=0.3)
    ax.set_title(f"rel_threshold={th} -> {n_clusters} clusters, {n_noise} noise pts")
plt.tight_layout()
plt.show()

## 6. Boundary / Bridge point detection

`watershed` มาตรฐานบังคับทุกจุดเข้ากลุ่มใดกลุ่มหนึ่งเสมอ แม้จุดนั้นจะอยู่กึ่งกลางพอดีระหว่างสองยอด (หุบเขา/สะพานเชื่อม) — demo นี้สร้างข้อมูลเฉพาะ: 2 blob ที่มีแนวจุดเรียงเชื่อมกันตรงกลาง (isthmus) เพื่อให้เห็น bridge point ชัดเจน (ต่างจาก `xy` synthetic data ด้านบนที่ blob แยกห่างกันจนแทบไม่มี bridge)

In [ ]:
rng = np.random.default_rng(7)
blob_a = rng.normal(loc=(0, 0), scale=0.5, size=(80, 2))
blob_b = rng.normal(loc=(6, 0), scale=0.5, size=(80, 2))
bridge_line = np.column_stack([
    np.linspace(1.5, 4.5, 20) + rng.normal(scale=0.15, size=20),
    rng.normal(scale=0.15, size=20),
])
bridge_xy = np.vstack([blob_a, blob_b, bridge_line])

bridge_clusterer = KDEWatershedClusterer(bw_method=0.2, grid_size=120, neighborhood_size=7, bridge_window=3)
bridge_labels = bridge_clusterer.fit_predict(bridge_xy)

n_core = int((bridge_labels >= 0).sum())
n_bridge = int((bridge_labels == -2).sum())
print(f"core points: {n_core} | bridge points: {n_bridge} | clusters: {len(bridge_clusterer.cluster_centers_)}")

fig, ax = plt.subplots(figsize=(7, 5))
ax.contourf(bridge_clusterer.kde_dict_["x_grid"], bridge_clusterer.kde_dict_["y_grid"], bridge_clusterer.kde_dict_["kde_values"], levels=20, cmap="Blues", alpha=0.6)
for cid in sorted(c for c in np.unique(bridge_labels) if c >= 0):
    pts = bridge_xy[bridge_labels == cid]
    ax.scatter(pts[:, 0], pts[:, 1], s=25, label=f"cluster {cid}", edgecolor="black", linewidth=0.3)
bpts = bridge_xy[bridge_labels == -2]
if len(bpts):
    ax.scatter(bpts[:, 0], bpts[:, 1], s=60, marker="D", facecolor="none", edgecolor="black", linewidth=1.5, label="bridge point (-2)")
ax.set_title(f"Bridge point detection (bridge_window={bridge_clusterer.bridge_window})")
ax.legend()
plt.show()

# ตัวอย่าง bridge_between ของจุดแรก ๆ ที่โดนจัดเป็น bridge (point index -> cluster id ที่อยู่ระหว่างกลาง)
list(bridge_clusterer.bridge_between_.items())[:5]

## Summary

- `KDEWatershedClusterer` = KDE (density surface) → peak finding (cluster seeds) → watershed พร้อม ridge line (segment ตาม "สันเขา" ของ density) → map จุดจริงกลับไปอ่าน label + ตรวจ boundary/bridge
- ไม่ต้องระบุจำนวน cluster ล่วงหน้า (ต่างจาก k-means) — จำนวน cluster มาจากจำนวน peak ที่รอดจาก `rel_threshold`
- จุดที่อยู่ระหว่างยอดเขาตั้งแต่ 2 ลูกขึ้นไป (หุบเขา/แอ่งน้ำ) ไม่ถูกยัดเข้ากลุ่มไหนอีกต่อไป ได้ `cluster_id = -2` พร้อม `bridge_between` บอกกลุ่มข้างเคียง — แยกความหมายจาก `-1` (ไม่มี peak เลยทั้ง dataset)
- Parameter หลักที่ต้อง tune: `bw_method` (ความเรียบของ density / จำนวน cluster), `rel_threshold` (ตัดกรอง noise/cluster ปลอม), `grid_size`/`neighborhood_size` (ความละเอียด vs. ความเร็ว), `bridge_window`/`bridge_rel_threshold` (ความไวการจับ bridge point — ยังไม่มีค่า default ที่ validate แล้ว ต้อง tune ตามข้อมูลจริง)
- ใช้จริงใน pipeline หลักที่ `lab/taxonomy_pipeline_demo.ipynb` โดย input `xy` คือ output ของ UMAP (`coordinate_xy`) แทน synthetic data ในนี้ — bridge point ที่นั่นถูกมองเป็น cross-reference ในสารบัญ ไม่ใช่ noise (ดู `docs/hierarchical-taxonomy-concept.md` section 10-11)